# dataclasses-replace-args — faded example 3: Zip two axes into paired (lr, batch_size) variants

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclasses-replace-args`. Running the beacon reports progress on the `Config: dataclasses.replace args` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: dataclasses.replace args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataclasses-replace-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataclasses-replace-args"
DD_SUBTOPIC = "Config: dataclasses.replace args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Unlike a Cartesian grid, a *paired* sweep walks two equal-length lists in lockstep with `zip`, producing one variant per pair. Each pair sets both fields in a single `dataclasses.replace(base, lr=lr, batch_size=bs)` call. The result length equals the shorter axis and base is never mutated.

## Faded exercise 3

### Faded — paired (lr, batch_size) sweep via zip

Implement `paired_sweep(base, lrs, batch_sizes)`. Walk `lrs` and `batch_sizes` together in lockstep (NOT a Cartesian product) and, for each pair, build one fresh `TrainingArgs` with BOTH `lr` and `batch_size` overridden in a single `replace` call. Return the variants in pair order. Complete the blanked line that builds each paired variant.

**Fill in:** Builds one new TrainingArgs from base with both lr and batch_size set from the current zipped pair, in a single dataclasses.replace call.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')

def paired_sweep(base, lrs, batch_sizes):
    out = []
    for lr, bs in zip(lrs, batch_sizes):
        variant = None  # TODO: new TrainingArgs from base with lr and bs both set in one replace call
        out.append(variant)
    return out


def _test():
    base = TrainingArgs()
    lrs = [1e-4, 3e-4, 1e-3]
    batch_sizes = [16, 32, 64]
    out = paired_sweep(base, lrs, batch_sizes)
    assert len(out) == 3
    for variant, lr, bs in zip(out, lrs, batch_sizes):
        assert isinstance(variant, TrainingArgs)
        assert variant.lr == lr
        assert variant.batch_size == bs
        assert variant.epochs == base.epochs
        assert variant.optimizer_name == base.optimizer_name
        assert variant is not base
    # paired, NOT cartesian: length == min axis
    short = paired_sweep(base, [1e-4, 2e-4], [8])
    assert len(short) == 1
    assert short[0].lr == 1e-4 and short[0].batch_size == 8
    # base not mutated
    assert base.lr == 1e-3 and base.batch_size == 32


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')

def paired_sweep(base, lrs, batch_sizes):
    out = []
    for lr, bs in zip(lrs, batch_sizes):
        variant = replace(base, lr=lr, batch_size=bs)
        out.append(variant)
    return out
```
</details>